In [1]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Digite sua GROQ_API_KEY: ")
print("✅ GROQ_API_KEY configurada!")

✅ GROQ_API_KEY configurada!


In [10]:
from typing import List
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# Definir a estrutura de dependências para recuperação
@dataclass
class RetrievalDependencies:
    documents: List[str]

# Definir o modelo de resultado esperado
class RAGResult(BaseModel):
    answer: str = Field(..., description="Resposta final gerada com base no contexto recuperado")
    retrieved_docs: List[str] = Field(..., description="Lista de documentos relevantes utilizados na resposta")

# Criar o agente RAG
rag_agent = Agent(
    'groq:openai/gpt-oss-120b',  # Substituído por modelo Groq que você já usa
    deps_type=RetrievalDependencies,
    output_type=RAGResult,
    system_prompt="Você é um assistente que utiliza informações recuperadas para responder com precisão. Use os documentos relevantes para compor sua resposta."
)

# Definir a ferramenta de recuperação de contexto
@rag_agent.tool()  # Adicionei parênteses aqui
async def retrieve_context(ctx: RunContext[RetrievalDependencies], query: str) -> List[str]:
    """
    Recupera documentos relevantes com base na consulta.
    
    Args:
        ctx: Contexto contendo os documentos disponíveis
        query: Consulta para filtrar documentos relevantes
        
    Returns:
        Lista de documentos que correspondem à consulta
    """
    # Busca simplificada: filtra documentos que contenham a query
    relevant = [doc for doc in ctx.deps.documents if query.lower() in doc.lower()]
    return relevant if relevant else ["Nenhum documento relevante encontrado."]

# Função principal adaptada para Jupyter
async def executar_rag():
    # Criar as dependências com documentos de exemplo
    dependencies = RetrievalDependencies(
        documents=[
            "O FastAPI é um framework moderno e rápido para construir APIs com Python, utilizando Pydantic para validação.",
            "Pydantic é uma biblioteca que fornece validação e tipagem baseada em Python, essencial para a robustez das aplicações.",
            "A integração entre FastAPI e Pydantic permite criar aplicações seguras e escaláveis, servindo de base para o PydanticAI."
        ]
    )
    
    # Executar o agente
    result = await rag_agent.run("Explique como o FastAPI se beneficia do uso do Pydantic.", deps=dependencies)
    
    print("Exemplo RAG:")
    print(f"Resposta: {result.output.answer}")
    print("\nDocumentos recuperados:")
    for i, doc in enumerate(result.output.retrieved_docs):
        print(f"{i+1}. {doc}")
    
    return result

# Em Jupyter, basta chamar com await - sem asyncio.run()
await executar_rag()

Exemplo RAG:
Resposta: O FastAPI tira proveito do Pydantic em várias frentes, o que o torna um dos frameworks mais produtivos e rápidos para criar APIs em Python:

1. **Validação automática de dados** – Quando você declara os tipos de entrada (por exemplo, parâmetros de caminho, query ou corpo de requisição) usando classes do Pydantic, o FastAPI valida automaticamente os dados recebidos. Se algum campo estiver ausente ou com tipo errado, o framework devolve uma resposta de erro 422 (Unprocessable Entity) contendo detalhes claros sobre o que falhou.

2. **Serialização e deserialização** – O Pydantic converte objetos Python para JSON (e vice‑versa) de forma transparente. Assim, ao retornar um modelo Pydantic, o FastAPI já entrega a representação JSON correta, sem necessidade de chamadas manuais como `json.dumps` ou `dict()`.

3. **Tipagem estática e autocompletar** – Como os modelos são declarados com anotações de tipo, IDEs e ferramentas de análise estática conseguem oferecer autocomple

AgentRunResult(output=RAGResult(answer='O FastAPI tira proveito do Pydantic em várias frentes, o que o torna um dos frameworks mais produtivos e rápidos para criar APIs em Python:\n\n1. **Validação automática de dados** – Quando você declara os tipos de entrada (por exemplo, parâmetros de caminho, query ou corpo de requisição) usando classes do Pydantic, o FastAPI valida automaticamente os dados recebidos. Se algum campo estiver ausente ou com tipo errado, o framework devolve uma resposta de erro\u202f422 (Unprocessable Entity) contendo detalhes claros sobre o que falhou.\n\n2. **Serialização e deserialização** – O Pydantic converte objetos Python para JSON (e vice‑versa) de forma transparente. Assim, ao retornar um modelo Pydantic, o FastAPI já entrega a representação JSON correta, sem necessidade de chamadas manuais como `json.dumps` ou `dict()`.\n\n3. **Tipagem estática e autocompletar** – Como os modelos são declarados com anotações de tipo, IDEs e ferramentas de análise estática c